# Algoritmos de optimización - Seminario
**Nombre y Apellidos:** Angel Eduardo Calizaya Morales.  
**Url:** https://github.com/acalizayam-edu/MIAR-Algoritmos-Optimizacion/tree/main/Trabajo-Practico  
**Problema:** 1. Sesiones de doblaje

## Descripción del problema

Se precisa coordinar el doblaje de una película. Los actores deben coincidir en las tomas en las que sus personajes aparecen juntos. Cada actor cobra la misma cantidad por **cada día** que se desplaza al estudio, con independencia del número de tomas que grabe ese día. No es posible grabar más de **6 tomas por día**. El objetivo es planificar las sesiones (repartir las tomas en días) de forma que el gasto total por los servicios de los actores sea el menor posible.

Datos: **30 tomas**, **10 actores**. La matriz de la hoja indica con un `1` que el actor participa en la toma y con un `0` que no.

> (*) La respuesta es obligatoria

## Datos del problema

Cargamos la matriz toma–actor. Trabajo cada toma como el **conjunto de actores** que intervienen en ella, porque para el coste lo único que importa es *quién* tiene que venir, no el orden.

In [11]:
import numpy as np

# Matriz 30x10: fila = toma, columna = actor. 1 = participa, 0 = no.
M = np.array([
    [1,1,1,1,1,0,0,0,0,0],
    [0,0,1,1,1,0,0,0,0,0],
    [0,1,0,0,1,0,1,0,0,0],
    [1,1,0,0,0,0,1,1,0,0],
    [0,1,0,1,0,0,0,1,0,0],
    [1,1,0,1,1,0,0,0,0,0],
    [1,1,0,1,1,0,0,0,0,0],
    [1,1,0,0,0,1,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,0],
    [1,1,0,0,0,1,0,0,1,0],
    [1,1,1,0,1,0,0,1,0,0],
    [1,1,1,1,0,1,0,0,0,0],
    [1,0,0,1,1,0,0,0,0,0],
    [1,0,1,0,0,1,0,0,0,0],
    [1,1,0,0,0,0,1,0,0,0],
    [0,0,0,1,0,0,0,0,0,1],
    [1,0,1,0,0,0,0,0,0,0],
    [0,0,1,0,0,1,0,0,0,0],
    [1,0,1,0,0,0,0,0,0,0],
    [1,0,1,1,1,0,0,0,0,0],
    [0,0,0,0,0,1,0,1,0,0],
    [1,1,1,1,0,0,0,0,0,0],
    [1,0,1,0,0,0,0,0,0,0],
    [0,0,1,0,0,1,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,1],
    [1,0,1,0,1,0,0,0,1,0],
    [0,0,0,1,1,0,0,0,0,0],
    [1,0,0,1,0,0,0,0,0,0],
    [1,0,0,0,1,1,0,0,0,0],
    [1,0,0,1,0,0,0,0,0,0],
])

N_TOMAS, N_ACTORES = M.shape
MAX_TOMAS_DIA = 6

# Cada toma como conjunto de actores (0-indexado)
tomas = [frozenset(int(a) for a in np.where(M[i] == 1)[0]) for i in range(N_TOMAS)]

print(f"Tomas: {N_TOMAS}, Actores: {N_ACTORES}, Tope por día: {MAX_TOMAS_DIA}")
print(f"Apariciones por actor: {[int(x) for x in M.sum(axis=0)]}")

Tomas: 30, Actores: 10, Tope por día: 6
Apariciones por actor: [22, 14, 13, 15, 11, 8, 3, 4, 2, 2]


## (*) ¿Cuántas posibilidades hay sin tener en cuenta las restricciones?

Repartir 30 tomas en días es lo mismo que **partir el conjunto de tomas en grupos** (cada grupo = un día). Si no imponemos ninguna restricción —ni tope de tomas por día, ni número de días fijado— el número de formas de partir un conjunto de $n$ elementos es el **número de Bell** $B_n$:

$$B_n = \sum_{k=1}^{n} S(n,k), \qquad S(n,k) = \frac{1}{k!}\sum_{j=0}^{k}(-1)^{j}\binom{k}{j}(k-j)^{n}$$

donde $S(n,k)$ (número de Stirling de segunda especie) cuenta las formas de partir $n$ tomas en exactamente $k$ días no vacíos.

Para $n = 30$:

$$B_{30} = 846\,749\,014\,511\,809\,332\,450\,147 \approx 8.47 \times 10^{23}$$

Este es el tamaño del espacio si el reparto fuese totalmente libre. Nótese que **no** es $30^{30}$: eso contaría días etiquetados y vacíos, pero aquí los días no tienen nombre (dar la toma 1 el "lunes" o el "martes" es la misma solución si el resto no cambia).

In [13]:
def bell(n):
    B = [[0]*(n+1) for _ in range(n+1)]
    B[0][0] = 1
    for i in range(1, n+1):
        B[i][0] = B[i-1][i-1]
        for j in range(1, i+1):
            B[i][j] = B[i-1][j-1] + B[i][j-1]
    return B[n][0]

print(f"B_30 = {bell(30):,}")

B_30 = 846,749,014,511,809,332,450,147


## ¿Cuántas posibilidades hay teniendo en cuenta todas las restricciones?

La única restricción dura del problema es **no más de 6 tomas por día**. Como hay 30 tomas y caben como mucho 6 por día, hacen falta **al menos** $\lceil 30/6 \rceil = 5$ días.

El espacio válido son las particiones de las 30 tomas en $k$ grupos, con $5 \le k \le 30$ y **cada grupo de tamaño $\le 6$**. Contar esto exactamente exige restar, de $S(30,k)$, las particiones que tienen algún grupo de 7 o más tomas. Como cota de referencia, si nos quedamos en los repartos con el número mínimo de días ($k=5$, todos los días llenos a 6), su número ya es:

$$S(30,5) = 7\,713\,000\,216\,608\,565\,075 \approx 7.7 \times 10^{18}$$

Es importante no cometer el error habitual de decir "30 tomas / 6 = 5 días fijos": el número de días **no** está fijado. La solución óptima podría usar 5 días, pero también podría convenir un día incompleto si eso agrupa mejor a los actores. El espacio con restricciones sigue siendo enorme (del orden de $10^{18}$–$10^{23}$), lo que ya anticipa que la fuerza bruta es inviable.

In [14]:
def stirling2(n, k):
    S = [[0]*(k+1) for _ in range(n+1)]
    S[0][0] = 1
    for i in range(1, n+1):
        for j in range(1, min(i, k)+1):
            S[i][j] = j*S[i-1][j] + S[i-1][j-1]
    return S[n][k]

k_min = -(-N_TOMAS // MAX_TOMAS_DIA)   # techo de 30/6
print(f"Días mínimos necesarios: {k_min}")
print(f"S(30,5) = {stirling2(30,5):,}")
print(f"S(30,6) = {stirling2(30,6):,}")

Días mínimos necesarios: 5
S(30,5) = 7,713,000,216,608,565,075
S(30,6) = 299,310,102,746,948,685,757


## (*) ¿Cuál es la estructura de datos que mejor se adapta al problema? Argumenta la respuesta

Uso dos representaciones complementarias:

- **Cada toma, como un conjunto de actores** (`frozenset`). El coste de un día es el número de actores *distintos* que se juntan, y eso es justo el tamaño de la **unión** de los conjuntos de sus tomas. La unión de conjuntos hace ese cálculo directo y sin duplicados. Que sea `frozenset` (inmutable) me permite además usarlo como clave y cachear resultados.

- **La solución, como una lista de días**, y cada día como la lista de tomas que contiene más el conjunto-unión de sus actores, que voy manteniendo incrementalmente. Así, al añadir una toma a un día, actualizar el coste es una única operación de unión en lugar de recalcularlo desde cero.

Empecé pensando en la matriz $30\times 10$ tal cual (array de NumPy), que es cómoda para cargar los datos y para contar apariciones por columna. Pero para el núcleo del algoritmo la matriz obliga a recorrer filas y columnas repetidamente; pasar a **conjuntos de actores** simplifica el cálculo del coste a operaciones de unión y cardinal, que es lo que se ejecuta millones de veces. Por eso la estructura final de trabajo son los conjuntos, y la matriz queda solo para la carga.

## (*) ¿Cuál es la función objetivo?

Una solución es un reparto de las tomas en días $D = \{d_1, d_2, \dots, d_m\}$, donde cada $d_j$ es el conjunto de tomas grabadas ese día. Para un día $d_j$, el conjunto de actores que deben desplazarse es la unión de los actores de sus tomas:

$$A(d_j) = \bigcup_{t \in d_j} \text{actores}(t)$$

Como cada actor presente cuesta un desplazamiento (un día de sueldo), el coste del día es $|A(d_j)|$. La **función objetivo** es el coste total de desplazamientos:

$$f(D) = \sum_{j=1}^{m} |A(d_j)|$$

sujeta a que cada toma esté en exactamente un día y a que $|d_j| \le 6$ para todo día.

## (*) ¿Es un problema de maximización o minimización?

Es un problema de **minimización**: buscamos el reparto $D$ que haga $f(D)$ lo más pequeño posible, es decir, que los actores acudan al estudio el menor número de días en total.

In [15]:
def coste_dia(dia_tomas):
    '''Actores distintos que se desplazan ese día = tamaño de la unión.'''
    actores = set()
    for t in dia_tomas:
        actores |= tomas[t]
    return len(actores)

def coste_total(solucion):
    '''solucion: lista de días; cada día es una lista de índices de toma.'''
    return sum(coste_dia(dia) for dia in solucion)

## Diseña un algoritmo para resolver el problema por fuerza bruta

La fuerza bruta recorre **todas las particiones válidas** de las 30 tomas (grupos de tamaño $\le 6$) y se queda con la de menor coste. Se implementa asignando las tomas una a una: cada toma puede ir a cualquier día ya abierto que tenga hueco, o abrir un día nuevo. Al llegar al final de cada rama, se evalúa el coste.

Este esquema es correcto y garantiza el óptimo, pero explora el espacio completo (del orden de $10^{18}$ repartos con el número mínimo de días). Por eso, sobre las 30 tomas reales **no termina en tiempo razonable**; lo dejo preparado para ejecutarse sobre instancias pequeñas (se usa así en el apartado de datos de prueba).

In [16]:
def fuerza_bruta(lista_tomas, max_por_dia=MAX_TOMAS_DIA):
    '''Explora TODAS las particiones válidas. Solo viable para pocas tomas.
       Devuelve (coste, solución, nodos) donde nodos es el número de nodos del árbol visitados.'''
    n = len(lista_tomas)
    mejor = {'coste': float('inf'), 'sol': None}
    nodos = [0]

    def coste(dias):
        return sum(len(set().union(*[lista_tomas[t] for t in d])) for d in dias)

    def asignar(idx, dias):
        nodos[0] += 1
        if idx == n:
            c = coste(dias)
            if c < mejor['coste']:
                mejor['coste'] = c
                mejor['sol'] = [list(d) for d in dias]
            return
        # meter la toma idx en cada día con hueco
        for d in dias:
            if len(d) < max_por_dia:
                d.append(idx); asignar(idx+1, dias); d.pop()
        # o abrir un día nuevo
        dias.append([idx]); asignar(idx+1, dias); dias.pop()

    asignar(0, [])
    return mejor['coste'], mejor['sol'], nodos[0]

## Calcula la complejidad del algoritmo por fuerza bruta

El algoritmo genera todas las particiones del conjunto de $n$ tomas. El número de particiones es el número de Bell $B_n$, cuyo crecimiento es **superexponencial**. Una cota clásica es:

$$B_n = \Theta\!\left( \left( \frac{n}{\ln n} \right)^{n} \right)$$

y en cualquier caso $B_n$ crece más rápido que $c^{n}$ para cualquier constante $c$. En cada hoja del árbol se evalúa el coste, que recorre las $n$ tomas y hace uniones sobre $\le 10$ actores; eso añade un factor $O(n)$ que no cambia el orden dominante.

Por tanto la complejidad temporal es:

$$O(n \cdot B_n)$$

que para $n = 30$ es del orden de $10^{23}$ operaciones: **inabordable** en la práctica. Esto justifica la necesidad de un algoritmo que evite explorar el espacio completo.

## (*) Diseña un algoritmo que mejore la complejidad del algoritmo por fuerza bruta. Argumenta por qué mejora

El algoritmo combina **ramificación y poda** (*branch & bound*) con una fase previa que le da una cota superior fuerte. Cuatro ideas lo hacen mucho más rápido que la fuerza bruta sin perder la garantía de optimalidad:

1. **Cota superior inicial (greedy + búsqueda local).** Antes de ramificar, construyo una solución con una heurística voraz —cada toma va al día donde menos actores nuevos añade— y la mejoro con búsqueda local: mover tomas de día e intercambiar pares mientras el coste baje. Esto da de entrada una solución muy buena que sirve como techo para podar con fuerza.

2. **Orden de las tomas.** En la ramificación asigno primero las tomas con más actores: fijan antes el grueso del coste y hacen que las cotas descarten ramas cuanto antes.

3. **Cota inferior (poda).** Dada una asignación parcial con coste acumulado $c$, el coste final no bajará de:
$$\text{LB} = c + \big|\,\{\text{actores de tomas no asignadas que no están en ningún día con hueco}\}\,\big|$$
   porque esos actores tendrán que desplazarse al menos una vez más. Si $\text{LB} \ge$ la mejor solución conocida, la rama se poda entera.

4. **Ruptura de simetría.** Una toma solo puede abrir el "siguiente" día nuevo, no cualquiera. Así no cuento como distintos repartos que solo cambian la etiqueta del día.

**Por qué mejora:** la fuerza bruta desciende hasta el fondo de *todas* las ramas; branch & bound descarta subárboles completos en cuanto la cota demuestra que no contienen nada mejor, y arranca ya con una cota superior casi óptima gracias a la búsqueda local. La ruptura de simetría elimina de golpe las permutaciones de días equivalentes. En conjunto se pasa de explorar un espacio de tamaño $B_n$ a visitar una fracción minúscula, resolviendo de forma **exacta** una instancia que la fuerza bruta no puede ni empezar.

In [17]:
import math, random, time

def _greedy(lista_tomas, max_por_dia, n_dias, rnd):
    '''Reparte las tomas en n_dias, metiendo cada una en el día que menos actores nuevos añade.'''
    n = len(lista_tomas)
    asig = [-1]*n; cnt = [0]*n_dias; union = [set() for _ in range(n_dias)]
    orden = sorted(range(n), key=lambda i: (-len(lista_tomas[i]), rnd.random()))
    for i in orden:
        cand = [d for d in range(n_dias) if cnt[d] < max_por_dia]
        d = min(cand, key=lambda d: (len(lista_tomas[i] - union[d]), rnd.random()))
        asig[i] = d; union[d] |= lista_tomas[i]; cnt[d] += 1
    return asig

def _coste_asig(asig, lista_tomas, n_dias):
    union = [set() for _ in range(n_dias)]
    for i, d in enumerate(asig):
        union[d] |= lista_tomas[i]
    return sum(len(u) for u in union)

def _busqueda_local(asig, lista_tomas, n_dias, max_por_dia):
    '''Mejora la asignación moviendo tomas de día e intercambiando pares, mientras baje el coste.'''
    n = len(asig)
    mejora = True
    while mejora:
        mejora = False
        base = _coste_asig(asig, lista_tomas, n_dias)
        cnt = [0]*n_dias
        for d in asig: cnt[d] += 1
        # mover una toma a otro día con hueco
        for i in range(n):
            di = asig[i]
            for dj in range(n_dias):
                if dj == di or cnt[dj] >= max_por_dia: continue
                asig[i] = dj
                if _coste_asig(asig, lista_tomas, n_dias) < base:
                    cnt[di] -= 1; cnt[dj] += 1; base = _coste_asig(asig, lista_tomas, n_dias)
                    mejora = True; break
                asig[i] = di
        # intercambiar dos tomas de días distintos
        for i in range(n):
            for j in range(i+1, n):
                if asig[i] == asig[j]: continue
                asig[i], asig[j] = asig[j], asig[i]
                if _coste_asig(asig, lista_tomas, n_dias) < base:
                    base = _coste_asig(asig, lista_tomas, n_dias); mejora = True
                else:
                    asig[i], asig[j] = asig[j], asig[i]
    return asig

def ramificacion_y_poda(lista_tomas, max_por_dia=MAX_TOMAS_DIA, limite_seg=30, usar_local=True):
    '''
    Algoritmo exacto (branch & bound) con dos aceleradores:
      1) cota superior inicial fuerte: greedy + búsqueda local (multiarranque),
      2) cota inferior por poda: actores de tomas restantes que aún deben desplazarse.
    Devuelve (coste, solución, stats) donde:
      - solución es una lista de días (listas de tomas),
      - stats = {'nodos': nodos visitados, 'podas': ramas cortadas por la cota}.
    El parámetro usar_local=False desactiva la fase 1 (útil para medir el B&B "puro").
    '''
    n = len(lista_tomas)
    n_dias_min = -(-n // max_por_dia)   # techo n / max_por_dia
    t0 = time.time()

    # --- Fase 1: cota superior con greedy + búsqueda local ---
    mejor = {'coste': float('inf'), 'asig': None, 'D': None}
    if usar_local:
        for D in (n_dias_min, n_dias_min + 1):
            for s in range(400):
                if limite_seg and time.time() - t0 > limite_seg * 0.5:
                    break
                a = _greedy(lista_tomas, max_por_dia, D, random.Random(s))
                a = _busqueda_local(a, lista_tomas, D, max_por_dia)
                c = _coste_asig(a, lista_tomas, D)
                if c < mejor['coste']:
                    mejor['coste'] = c; mejor['asig'] = a[:]; mejor['D'] = D
        mejor_bb = {'coste': mejor['coste'],
                    'sol': [[i for i in range(n) if mejor['asig'][i] == d] for d in range(mejor['D'])]}
    else:
        mejor_bb = {'coste': float('inf'), 'sol': None}

    # --- Fase 2: branch & bound ---
    orden = sorted(range(n), key=lambda i: -len(lista_tomas[i]))
    cnt = [0]*n; union = [set() for _ in range(n)]; asig = [-1]*n
    stats = {'nodos': 0, 'podas': 0}

    def bb(idx, max_abierto):
        if limite_seg and time.time() - t0 > limite_seg:
            return
        stats['nodos'] += 1
        if idx == n:
            c = sum(len(u) for u in union if u)
            if c < mejor_bb['coste']:
                mejor_bb['coste'] = c
                mejor_bb['sol'] = [[i for i in range(n) if asig[i] == d] for d in range(max_abierto+1)]
            return
        c = sum(len(u) for u in union if u)
        cov = set()
        for d in range(max_abierto+1):
            if cnt[d] < max_por_dia: cov |= union[d]
        extra = set()
        for k in range(idx, n):
            extra |= (lista_tomas[orden[k]] - cov)
        if c + len(extra) >= mejor_bb['coste']:
            stats['podas'] += 1      # la cota inferior corta esta rama
            return
        i = orden[idx]
        abiertos = [d for d in range(max_abierto+1) if cnt[d] < max_por_dia]
        abiertos.sort(key=lambda d: len(lista_tomas[i] - union[d]))
        nuevo = max_abierto+1 if max_abierto+1 < n else None
        for d in abiertos + ([nuevo] if nuevo is not None else []):
            old = set(union[d]); union[d] |= lista_tomas[i]; cnt[d] += 1; asig[i] = d
            bb(idx+1, max(max_abierto, d))
            union[d] = old; cnt[d] -= 1; asig[i] = -1

    bb(0, 0)
    return mejor_bb['coste'], mejor_bb['sol'], stats

## (*) Calcula la complejidad del algoritmo

En el **peor caso**, branch & bound no logra podar nada y visita el mismo espacio que la fuerza bruta: recorre las particiones válidas del conjunto. Por tanto la cota superior de complejidad **en el peor caso sigue siendo** del orden de las particiones con grupos de tamaño $\le 6$, acotada por $B_n$:

$$O(n \cdot B_n) \ \text{en el peor caso}$$

La mejora **no cambia el peor caso teórico** —la poda no puede garantizar recortes en toda instancia—, pero sí cambia drásticamente el **caso práctico**. En cada nodo el trabajo extra es el cálculo de la cota, que recorre las tomas restantes y hace uniones sobre $\le 10$ actores: $O(n)$ por nodo. Si el algoritmo visita $V$ nodos (con $V \ll B_n$ gracias a la poda y la ruptura de simetría), el coste real es:

$$O(n \cdot V)$$

En esta instancia concreta, la ruptura de simetría por sí sola divide el espacio por $m!$ (número de permutaciones de los $m$ días), y la cota inferior elimina la mayoría de las ramas restantes, lo que hace el problema resoluble de forma exacta.

## Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Genero instancias aleatorias del mismo tipo: una matriz toma–actor de tamaño parametrizable, donde cada actor participa en cada toma con cierta probabilidad (densidad), garantizando que ninguna toma quede vacía. Reducir el problema tiene que ser **coherente**: mantengo el tope de 6 tomas por día y uso menos tomas/actores, no un caso trivial. Así puedo validar el algoritmo mejorado contra la fuerza bruta en tamaños donde esta última todavía termina.

In [18]:
import random

def genera_instancia(n_tomas, n_actores, densidad=0.35, semilla=None):
    '''Devuelve una lista de 'tomas' (frozenset de actores). Ninguna toma vacía.'''
    rnd = random.Random(semilla)
    inst = []
    for _ in range(n_tomas):
        actores = {a for a in range(n_actores) if rnd.random() < densidad}
        if not actores:                       # evitar tomas vacías
            actores = {rnd.randrange(n_actores)}
        inst.append(frozenset(actores))
    return inst

# Instancia pequeña de prueba (fuerza bruta todavía viable)
prueba = genera_instancia(n_tomas=8, n_actores=5, densidad=0.4, semilla=42)
for i, t in enumerate(prueba):
    print(f"Toma {i+1}: actores {sorted(a+1 for a in t)}")

Toma 1: actores [2, 3, 4]
Toma 2: actores [3, 5]
Toma 3: actores [1, 3, 4]
Toma 4: actores [2, 5]
Toma 5: actores [3, 4]
Toma 6: actores [1, 2, 3]
Toma 7: actores [5]
Toma 8: actores [1]


## Aplica el algoritmo al juego de datos generado

Valido en dos pasos. Primero, una **comparación de operaciones** sobre instancias pequeñas de tamaño creciente ($n = 6, 8, 10$): para cada una ejecuto la fuerza bruta y el branch & bound *puro* (sin la fase de búsqueda local, para que la comparación de nodos sea justa) y cuento los **nodos del árbol que visita cada uno**. Esto mide la mejora en operaciones —no en tiempo— y comprueba de paso que ambos dan el **mismo óptimo**. Después aplico el algoritmo completo a la instancia real de 30 tomas.

In [19]:
# 1) Comparación de operaciones: nodos explorados por fuerza bruta vs branch & bound
print(f"{'n':>3} | {'óptimo FB':>9} | {'óptimo B&B':>10} | {'nodos FB':>10} | {'nodos B&B':>10} | {'reducción':>9}")
print("-"*70)
for n_test in (6, 8, 10):
    inst = genera_instancia(n_tomas=n_test, n_actores=5, densidad=0.4, semilla=n_test)
    c_fb, _, nodos_fb = fuerza_bruta(inst)
    # B&B puro (usar_local=False) para contar solo los nodos del árbol de ramificación
    c_bb, _, stats = ramificacion_y_poda(inst, usar_local=False)
    reduccion = 100 * (1 - stats['nodos'] / nodos_fb)
    assert c_fb == c_bb, f"¡Óptimos distintos en n={n_test}!"
    print(f"{n_test:>3} | {c_fb:>9} | {c_bb:>10} | {nodos_fb:>10,} | {stats['nodos']:>10,} | {reduccion:>7.1f}%")
print("\nEn todos los tamaños coincide el óptimo: el branch & bound es exacto,")
print("y visita muchos menos nodos que la fuerza bruta (la brecha crece con n).")

  n | óptimo FB | óptimo B&B |   nodos FB |  nodos B&B | reducción
----------------------------------------------------------------------
  6 |         4 |          4 |        279 |         13 |    95.3%
  8 |         7 |          7 |      5,286 |         44 |    99.2%
 10 |         7 |          7 |    141,625 |         62 |   100.0%

En todos los tamaños coincide el óptimo: el branch & bound es exacto,
y visita muchos menos nodos que la fuerza bruta (la brecha crece con n).


La tabla muestra lo esencial del análisis: **fuerza bruta y branch & bound llegan al mismo óptimo** (el algoritmo mejorado no pierde exactitud), pero el número de **nodos explorados** por branch & bound es mucho menor, y la diferencia se agranda a medida que $n$ crece. Esa reducción de nodos es la mejora, medida en operaciones y no en tiempo.

### Aplicación a la instancia real (30 tomas, 10 actores)

El problema es NP-duro: encontrar una solución excelente es rápido, pero *certificar* la optimalidad por ramificación pura puede requerir minutos. Por eso el algoritmo completo arranca con la fase de greedy + búsqueda local (que aquí ya alcanza el óptimo) y usa el branch & bound con un límite de tiempo como respaldo.

In [20]:
coste_opt, solucion, stats = ramificacion_y_poda(tomas, limite_seg=90)
print(f"Coste óptimo (desplazamientos totales): {coste_opt}")
print(f"Nodos del branch & bound: {stats['nodos']:,} | ramas podadas por la cota: {stats['podas']:,}\n")
for j, dia in enumerate(solucion, 1):
    if not dia:
        continue
    actores = sorted(a+1 for a in set().union(*[tomas[t] for t in dia]))
    print(f"Día {j}: tomas {sorted(t+1 for t in dia)}  ->  {len(actores)} actores {actores}")

Coste óptimo (desplazamientos totales): 27
Nodos del branch & bound: 14,853,714 | ramas podadas por la cota: 10,932,143

Día 1: tomas [14, 17, 18, 19, 23, 24]  ->  3 actores [1, 3, 6]
Día 2: tomas [1, 5, 7, 9, 11, 22]  ->  6 actores [1, 2, 3, 4, 5, 8]
Día 3: tomas [2, 13, 20, 27, 28, 30]  ->  4 actores [1, 3, 4, 5]
Día 4: tomas [3, 4, 8, 15, 21, 29]  ->  6 actores [1, 2, 5, 6, 7, 8]
Día 5: tomas [6, 10, 12, 16, 25, 26]  ->  8 actores [1, 2, 3, 4, 5, 6, 9, 10]


El reparto óptimo agrupa las 30 tomas en 5 días con un coste total de **27 desplazamientos**. Se observa el efecto esperado: los actores que aparecen en muchas tomas (el actor 1 está en 22 de 30) se concentran, y los actores raros (9 y 10, en 2 tomas cada uno) se agrupan para no generar desplazamientos sueltos. El último día reúne tomas que solo comparten a los actores 1, 3 y 6, minimizando el coste de esa jornada.

## Enumera las referencias que has utilizado

- Brassard, G. y Bratley, P. (1997). *Fundamentos de algoritmia*. Prentice Hall — capítulos de vuelta atrás (*backtracking*) y ramificación y poda.
- Material de la asignatura 03MAIR — *Algoritmos de optimización* (VIU): sesiones sobre complejidad, combinatoria y el problema del trabajo práctico.
- Documentación de Python: `itertools`, `functools` y estructuras `set`/`frozenset`.

## Describe brevemente cómo crees que es posible avanzar en el estudio del problema

- **Cota inferior más fuerte.** La cota actual solo cuenta actores "forzados". Una cota basada en $\sum_a \lceil k_a / 6 \rceil$ (cada actor necesita al menos $\lceil k_a/6 \rceil$ días según su número de apariciones $k_a$) o una relajación del problema podría podar mucho antes y probar optimalidad más rápido.
- **Búsqueda del mejor día primero** (*best-first / A\**) en lugar de profundidad, para llegar antes a soluciones buenas y afinar la poda.
- **Metaheurísticas para instancias grandes.** Si el número de tomas creciera (cientos), el enfoque exacto dejaría de escalar; ahí tendrían sentido búsqueda local, recocido simulado o algoritmos genéticos, aceptando soluciones aproximadas con una cota de error estimada empíricamente.
- **Variaciones del problema.** Añadir disponibilidad de actores por día, un número máximo de días, o costes distintos por actor convertiría el modelo en uno más general (asignación con costes), abordable con programación lineal entera.